# Pipeline Huấn Luyện 2-Stage (Waste Detection)
Notebook này được thiết kế để chạy trên **Kaggle** với GPU P100 (hoặc T4 x2).

### Yêu cầu chuẩn bị dữ liệu trên Kaggle:
Trước khi chạy notebook này, hãy đảm bảo bạn đã Add Data (Datasets) các thành phần sau vào Kaggle:
1. **Code Đồ Án**: Nén toàn bộ thư mục project (chứa thư mục `src`, `models`, `data`...) thành file zip và upload lên Kaggle Dataset (ví dụ tên là `waste-project-code`).
2. **Dataset Stage 1 (YOLO Binary)**: Upload thư mục `data/processed_binary_tiled` (hoặc `processed_binary` nếu không dùng tiling).
3. **Dataset Stage 2 (Classification Merged)**: Chạy script `merge_external_datasets.py` ở Local để gộp TACO + TrashNet + RealWaste, nén thư mục output và upload lên Kaggle.

**LƯU Ý:** Đảm bảo bạn bật chế độ **Internet** và **GPU** trong phần Settings bên phải của Kaggle.

## Bước 1: Setup Môi Trường & Code

In [ ]:
!pip install -q ultralytics timm torch torchvision

import os
import shutil
from pathlib import Path

# ---------------------------------------------------------
# CHÚ Ý: Sửa đường dẫn bên dưới trỏ tới Kaggle Dataset chứa code của bạn.
# Ví dụ: /kaggle/input/waste-project-code/waste-detection_project
# ---------------------------------------------------------
CODE_INPUT_DIR = "/kaggle/input/waste-project-code" 
WORKING_DIR = "/kaggle/working/project"

# Copy code từ thư mục read-only của Kaggle sang /kaggle/working để có thể chạy và sửa
if not os.path.exists(WORKING_DIR):
    shutil.copytree(CODE_INPUT_DIR, WORKING_DIR)
    print(f"Đã copy source code sang {WORKING_DIR}")

# Chuyển thư mục làm việc hiện tại vào project
os.chdir(WORKING_DIR)
print("Current working directory:", os.getcwd())

# Hiển thị cấu trúc thư mục để kiểm tra
!ls -la

## Bước 2: Chạy Huấn Luyện Stage 1 (YOLO Binary Detection)
Chúng ta sẽ thay đổi flag `ON_KAGGLE = True` trong code tự động và tiến hành train.

In [ ]:
stage1_script = "src/train_stage1_detector.py"

# Đọc file code và tự động bật cờ ON_KAGGLE = True
with open(stage1_script, "r", encoding="utf-8") as f:
    code = f.read()

code = code.replace("ON_KAGGLE = False", "ON_KAGGLE = True")

# Thay đổi đường dẫn dataset input cho Kaggle (Sửa lại tên dataset cho đúng với của bạn)
code = code.replace(
    'DATA_DIR    = Path("/kaggle/input/waste-binary-tiled")',
    'DATA_DIR    = Path("/kaggle/input/dataset-yolo-binary-waste")' # Sửa tên Dataset ở đây
)

with open(stage1_script, "w", encoding="utf-8") as f:
    f.write(code)

print("Đã cấu hình script Stage 1 cho Kaggle.")

In [ ]:
# Chạy huấn luyện YOLO (Thời gian chạy tùy thuộc vào số Epoch, thường mất 1-2 tiếng)
!python src/train_stage1_detector.py

## Bước 3: Chạy Huấn Luyện Stage 2 (EfficientNet Classification)
Mô hình sẽ xử lý mất cân bằng dữ liệu và huấn luyện 50 epoch.

In [ ]:
stage2_script = "src/train_stage2_classifier.py"

with open(stage2_script, "r", encoding="utf-8") as f:
    code = f.read()

code = code.replace("ON_KAGGLE = False", "ON_KAGGLE = True")

# Thay đổi đường dẫn dataset input cho Kaggle (Sửa lại tên dataset classification merged)
code = code.replace(
    'DATA_DIR = Path("/kaggle/input/waste-classification-merged")',
    'DATA_DIR = Path("/kaggle/input/dataset-classification-merged")' # Sửa tên Dataset ở đây
)

with open(stage2_script, "w", encoding="utf-8") as f:
    f.write(code)

print("Đã cấu hình script Stage 2 cho Kaggle.")

In [ ]:
# Chạy huấn luyện EfficientNet-B2
!python src/train_stage2_classifier.py

## Bước 4: Lưu trữ kết quả (Download Weights)
Kaggle sẽ tự động đóng gói toàn bộ thư mục `/kaggle/working` thành file output khi bạn ấn nút **Save Version -> Save & Run All (Commit)**. Tuy nhiên, nếu bạn chạy thủ công từng cell, bạn có thể tải trực tiếp 2 file trọng số quan trọng nhất về.

In [ ]:
import IPython
from IPython.display import FileLink

stage1_weight = "/kaggle/working/stage1_best.pt"
stage2_weight = "/kaggle/working/stage2_best.pth"

if os.path.exists(stage1_weight):
    print("Tải weights YOLO (Stage 1):")
    display(FileLink(stage1_weight))
else:
    print("Chưa có weights YOLO.")

if os.path.exists(stage2_weight):
    print("Tải weights EfficientNet (Stage 2):")
    display(FileLink(stage2_weight))
else:
    print("Chưa có weights EfficientNet.")